<a href="https://colab.research.google.com/github/itgirlhightech/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [6]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [21]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.547     0.337     0.417      9389
           1      0.685     0.838     0.754     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.587     0.585     25551
weighted avg      0.634     0.654     0.630     25551



TASK DONE:
After adding imp_last30, the model accuracy increased to over 90%, which indicates data leakage. This feature is directly related to the target (is_declining), allowing the model to indirectly access the answer. After removing the feature, the performance returned to a more realistic level, confirming that the previous result was artificially inflated by leakag

The model slightly outperforms the majority-class baseline (65.7% accuracy versus a 63.3% baseline). It performs considerably better on the declining-content class (1), achieving high recall (0.842) and an F1-score of 0.756. This means the model successfully identifies most pages that are actually declining, which is valuable for prioritizing content refresh. However, performance on class 0 is weaker, indicating that the model tends to classify many pages as declining even when they are not.

# 1. What does one row mean?

One row represents the search performance of a content page over a defined time window. It contains metrics such as impressions, clicks, CTR, and avarage position that describe how the content performed during that period.

# 2. Which tables will you use?

I will primarily use fact_content_query_90d because it contains the performance metrics needed for my lane. I will also use the dimension tables (dim_content and dim_clients) to enrich the data with descriptive information when neessary.

# 3. Which time window?

I will use a mid-panel month (2026-03) and the available 90-day performance window to build features while avoiding the final month reserved for evaluation.

#4. What would you predict or rank?

I want to rank content pages by their Opportunity Score, which acts as a proxy for refresh potencial. The goal is to prioritize pages that are most likely to benefit from a content update.

#5. One thing you deliberaly exclude

I deliberately exclude the final month of data because it should remain an unseen evaluation period and using it could introduce data leakage.

In [11]:
#Grain

grain = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    query_hash_id,
    window_start,
    window_end,
    COUNT(*) AS rows_per_grain
FROM {TABLES['fact_query_90d']}
GROUP BY
    client_hash_id,
    content_hash_id,
    query_hash_id,
    window_start,
    window_end
HAVING COUNT(*) > 1
""")

grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬───────────────┬──────────────┬────────────┬────────────────┐
│ client_hash_id │ content_hash_id │ query_hash_id │ window_start │ window_end │ rows_per_grain │
│    varchar     │     varchar     │    varchar    │     date     │    date    │     int64      │
├────────────────┴─────────────────┴───────────────┴──────────────┴────────────┴────────────────┤
│                                            0 rows                                             │
└───────────────────────────────────────────────────────────────────────────────────────────────┘

In [12]:
#row count + date span

row_stats = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(window_start) AS first_window,
    MAX(window_end) AS last_window
FROM {TABLES['fact_query_90d']}
""")

row_stats

┌───────────┬──────────────┬─────────────┐
│ row_count │ first_window │ last_window │
│   int64   │     date     │    date     │
├───────────┼──────────────┼─────────────┤
│   2414248 │ 2026-04-02   │ 2026-06-30  │
└───────────┴──────────────┴─────────────┘

In [16]:
# Availability

availability = con.sql(f"""
SELECT
      COUNT(*) AS available_clients
FROM {TABLES['fact_daily']}
WHERE ga4_data_available IS TRUE
""")

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┐
│ available_clients │
│       int64       │
├───────────────────┤
│           2816455 │
└───────────────────┘

In [19]:
# Feature frame

features = con.sql(f"""
SELECT
      client_hash_id,
      content_hash_id,
      impressions_90d,
      clicks_90d,
      avg_position_90d,
      content_visible_query_count,
      rare_impressions_share
FROM {TABLES['fact_query_90d']}
LIMIT 10
""")
features

┌─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬────────────────────┬─────────────────────────────┬────────────────────────┐
│     client_hash_id      │     content_hash_id      │ impressions_90d │ clicks_90d │  avg_position_90d  │ content_visible_query_count │ rare_impressions_share │
│         varchar         │         varchar          │      int64      │   int64    │       double       │            int64            │         double         │
├─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼────────────────────┼─────────────────────────────┼────────────────────────┤
│ client_08a6a72ff48e62c0 │ content_447894f2faf0d2bc │              11 │          0 │ 10.818181818181818 │                          14 │    0.04365620736698499 │
│ client_08a6a72ff48e62c0 │ content_447894f2faf0d2bc │              13 │          0 │ 1.7692307692307692 │                          14 │    0.04365620736698499 │
│ client_08a6a72ff48e62c0 │ 

## 1. Impressions_90d

Knowable at the decision moment because it summarizes the content's search visibility over the previous 90 days, before any refresh decision is made.

## 2. Clicks_90d

Knowable at the decision moment because click is already available from past Search Console performance.

## 3. Avg_position_90d

Knowable at the decision moment because it counts the queries that already generated impressions for the content.

## 4. Content_visible_query_count

Knowable at the decision moment because it counts the queries that already generated impressions for the content.

## 5. Rare_impressions_share

Knowable at the decision moment because it is computed from historical search impressions and does not rely on future performance.


# One named limitation of your slice

This slice is limited to a 90-day observation window, which may not capture long-term seasonal patterns or content performance changes that occur over longer periods.

#  Self-check

I completed all the required steps, including the data contract, verification queries, feature engineering, and the leakage experiment. Running every cell helped me understand the workflow, and the leakage experiment showed how a single feature can dramatically inflate model performance by introducing information from the target. This reinforced the importance of validating features before training a model.